# imports

In [1]:
# !git clone https://github.com/moryacho/street-pattern-classifier.git

# !pip install huggingface_hub geopandas osmnx
# !pip install torch_geometric
# !pip install "folium>=0.12" matplotlib mapclassify

# !pip uninstall torch torchvision torchaudio -y
# !pip cache purge
# !pip install torch torchvision torchaudio

import torch

In [2]:
import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
import time
from tqdm import tqdm
import os
import networkx as nx
import json

import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats
import warnings

from shapely.geometry import LineString, Polygon, Point
import shapely
import pickle
import ast



# city name to osm id

In [ ]:
osm_city_names = [
    # East Asia
    # "Beijing, China",
    # "Chengdu, China",
    # "Chongqing, China",
    # "Guangzhou, China",
    # "Hangzhou, China",
    # "Shanghai, China",
    # "Shenzhen, China",
    # "Wuhan, China",
    # "Nanjing, China",
    # "Taipei, Taiwan",
    # "Tokyo, Japan", # multyoplygon: so buffer in the ocean
    # "Osaka, Japan",
    # "Seoul, South Korea",
    # "Tianjin, China",
    
    # South Asia
    # "Bangkok, Thailand",
    # "Delhi, India",
    # "Singapore, Singapore", # none
    # "Kuala Lumpur, Malaysia",
    # "Mumbai, India",
    # "Hyderabad, India",
    # "Kathmandu, Nepal",
    
    # Oceania
    # "Melbourne, Australia",
    # "Sydney, Australia",
    # "Auckland, New Zealand",
    # "Brisbane, Australia",
    
    # Africa
    # "Cairo, Egypt", # none
    "Casablanca, Morocco",
    "Lagos, Nigeria",
    "Nairobi, Kenya",
    "Johannesburg, South Africa",
    
    # Europe
    "Amsterdam, Netherlands",
    "Brussels, Belgium",
    "London, United Kingdom",
    "Paris, France",
    "Berlin, Germany",
    "Munich, Germany",
    "Madrid, Spain",
    "Milan, Italy",
    "Moscow, Russia",
    "Stockholm, Sweden",
    "Bucharest, Romania",
    "Saint Petersburg, Russia",
    
    # North America
    "New York City, USA",  # Важно: именно "New York City", а не просто "New York"
    "Los Angeles, USA",
    "Detroit, USA",
    "Philadelphia, USA",
    "San Antonio, USA",
    "San Diego, USA",
    "Toronto, Canada",
    "Montreal, Canada",
    "Chicago, USA",
    "Houston, USA",
    "Phoenix, USA",
    
    # South America
    "Lima, Peru",
    "Bogota, Colombia",
    "Sao Paulo, Brazil",  # Можно также "São Paulo, Brazil"
    "Caracas, Venezuela",
    "Brasilia, Brazil",    # Можно также "Brasília, Brazil"
    "Asuncion, Paraguay"
]

In [3]:
import osmapi as osm
api = osm.OsmApi()

In [29]:
relation = api.relation_get(17140517)
relation

{'id': 17140517,
 'visible': True,
 'version': 5,
 'changeset': 175450670,
 'timestamp': datetime.datetime(2025, 12, 3, 14, 55),
 'user': 'Aleksandar Matejevic',
 'uid': 8724788,
 'tag': {'name': 'Singapore',
  'name:ace': 'Singapura',
  'name:af': 'Singapoer',
  'name:als': 'Singapur',
  'name:am': 'ሲንጋፖር',
  'name:an': 'Singapur',
  'name:ar': 'سنغافورة',
  'name:arz': 'سينجابوره',
  'name:ast': 'Singapur',
  'name:az': 'Sinqapur',
  'name:az-Arab': 'سنقاپور',
  'name:azb': 'سنقاپور',
  'name:ba': 'Сингапур',
  'name:bar': 'Singapur',
  'name:bat-smg': 'Singapūrs',
  'name:bcl': 'Singapore',
  'name:be': 'Сінгапур',
  'name:be-tarask': 'Сынгапур',
  'name:bg': 'Сингапур',
  'name:bi': 'Singapore',
  'name:bjn': 'Singapura',
  'name:bn': 'সিঙ্গাপুর',
  'name:bo': 'སེང་ག་ཕོར།',
  'name:bpy': 'সিঙ্গাপুর',
  'name:br': 'Singapoura',
  'name:bs': 'Singapur',
  'name:bug': 'Singapura',
  'name:bxr': 'Сингапур',
  'name:ca': 'Singapur',
  'name:cdo': 'Sĭng-gă-pŏ̤',
  'name:ce': 'Сингапур',


In [ ]:
node = api.node_get(1889910974)
node

{'id': 16173236,
 'visible': True,
 'version': 81,
 'changeset': 166253295,
 'timestamp': datetime.datetime(2025, 5, 14, 12, 23, 39),
 'user': 'Alex202210',
 'uid': 9345234,
 'lat': 28.6138954,
 'lon': 77.2090057,
 'tag': {'admin_level': '2',
  'alt_name:ar': 'نيودلهي',
  'alt_name:ks': 'نٔو دِل',
  'alt_name:ur': 'نئی دلی',
  'capital': 'yes',
  'is_capital': 'country',
  'name': 'New Delhi',
  'name:ace': 'New Delhi',
  'name:af': 'Nieu-Delhi',
  'name:am': 'ኒው ዴሊ',
  'name:an': 'Nueva Delhi',
  'name:ang': 'Nīƿe Delhi',
  'name:ar': 'دلهي الجديدة',
  'name:as': 'নতুন দিল্লী',
  'name:az': 'Yeni Dehli',
  'name:az-Arab': 'یئنی دهلی',
  'name:azb': 'یئنی دهلی',
  'name:ba': 'Нью-Дели',
  'name:bat-smg': 'Naujasės Delės',
  'name:be': 'Нью-Дэлі',
  'name:be-tarask': 'Нью-Дэлі',
  'name:bg': 'Ню Делхи',
  'name:bh': 'नई दिल्ली',
  'name:bn': 'নতুন দিল্লি',
  'name:bo': 'ནེའུ་དིལ་ལི།',
  'name:bpy': 'নুৱা দিল্লী',
  'name:br': 'New Delhi',
  'name:bs': 'New Delhi',
  'name:ca': 'Nova Del

In [41]:
BUFFER = 20000
big_crs = 4326
node_id = 25730724

node = api.node_get(node_id)


p = shapely.Point(node["lon"], node["lat"])
g = gpd.GeoDataFrame({"geometry" : [p]}, geometry="geometry", crs=4326)
g_proj = g.to_crs(g.estimate_utm_crs())
point_gdf_utm = gpd.GeoDataFrame(geometry=[g_proj.iloc[0].geometry], crs=g_proj.crs)
buffer_utm = point_gdf_utm.buffer(BUFFER)
buffer_geo = buffer_utm.to_crs(g_proj.crs)    

geom_polygon = buffer_geo.to_crs(big_crs)


p = shapely.Point(node["lon"], node["lat"])

gdf_to_save = gpd.GeoDataFrame({"city_name":["Casablanca, Morocco"],
                        "node_osm_id" : [node_id],
                        "node" : [p],
                        "geometry" : [geom_polygon.iloc[0]],
                        "graph_2020" : [query_year(geom_polygon.iloc[0], 2020)],
                        "graph_2024" : [query_year(geom_polygon.iloc[0], 2024)]},
                        geometry="geometry",
                        crs=4326)

gdf_to_save.to_pickle("graphs_by_city//Casablanca_Morocco.pkl")

In [6]:
# Source - https://stackoverflow.com/a/77965352
# Posted by Nick ODell
# Retrieved 2026-03-19, License - CC BY-SA 4.0

import osmnx as ox

# Must be an overpass instance which supports attic
ox.settings.overpass_endpoint = "https://overpass-api.de/api"

def query_year(coordinate, year):
    date = f'{year}-01-01T00:00:00Z'
    # Request attic data
    ox.settings.overpass_settings = '[out:json][timeout:{timeout}][date:"' + date + '"]{maxsize}'
    graph = ox.graph.graph_from_polygon(coordinate)
    # Restore old settings
    ox.settings.overpass_settings = '[out:json][timeout:{timeout}]{maxsize}'
    return graph


In [81]:
# p = shapely.Point(node["lon"], node["lat"])

# gdf_to_save = gpd.GeoDataFrame({"city_name":["Taipei, Taiwan"],
#                         "node_osm_id" : [1147314253],
#                         "node" : [p],
#                         "geometry" : [geom_polygon.iloc[0]],
#                         "graph_2020" : [query_year(geom_polygon.iloc[0], 2020)],
#                         "graph_2024" : [query_year(geom_polygon.iloc[0], 2024)]},
#                         geometry="geometry",
#                         crs=4326)

In [ ]:
# gdf_to_save.to_pickle("graphs_by_city//Taipei_Taiwan.pkl")

In [77]:
gdf_to_save

,city_name,node_osm_id,node,geometry,graph_2020,graph_2024
0,"Nanjing, China",244081381,POINT (118.7788631 32.0438284),"POLYGON ((118.99059 32.04068, 118.98918 32.023...","(29571632, 29571641, 29571710, 32918369, 30481...","(29571632, 29571639, 29571641, 29571709, 30481..."
